# Hyperliquid 测试网演示

演示在 Hyperliquid 测试网上的常见操作：市场数据查询、账户查询、限价挂单与撤单、市价开仓与平仓。

密钥从仓库根目录的 `.env` 读取（`HL_ACC_ADD` 钱包地址、`HL_SECRET_KEY` 私钥），不在代码中写入明文。`HL_ACC_ADD` 可能是主钱包，也可能是代理（API/agent）钱包；若是代理钱包，资金与订单都归属它背后的主账户，下方会自动解析出主账户地址用于查询。测试网用的是水龙头发放的 mock USDC，下单会真实成交。

In [1]:
import os
from pathlib import Path

from dotenv import load_dotenv
import eth_account
import pandas as pd

from hyperliquid.info import Info
from hyperliquid.exchange import Exchange
from hyperliquid.utils import constants

ROOT = Path.cwd()
while not (ROOT / ".env").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
load_dotenv(ROOT / ".env")

env_address = os.environ["HL_ACC_ADD"]
account = eth_account.Account.from_key(os.environ["HL_SECRET_KEY"])

info = Info(constants.TESTNET_API_URL, skip_ws=True)

# 若 env 地址是代理钱包, 资金和订单归属其背后的主账户, 查询须用主账户地址
role = info.post("/info", {"type": "userRole", "user": env_address})
if role["role"] == "agent":
    ADDRESS = role["data"]["user"]
    print("env 地址为代理钱包, 解析出主账户:", ADDRESS)
else:
    ADDRESS = env_address
    print("env 地址为主钱包:", ADDRESS)

exchange = Exchange(account, constants.TESTNET_API_URL, account_address=ADDRESS)
print("签名钱包:", account.address)

env 地址为代理钱包, 解析出主账户: 0x2a6eee373827f987661eed4d8bf2e215102dae45


签名钱包: 0x89E9273ff4F659F4B178b50315751D34d4120b08


## 1. 市场数据查询

查询合约元数据、各币种中间价、ETH 订单簿前几档，以及最近 24 根 1h K 线。

In [2]:
meta = info.meta()
universe = meta["universe"]
print(f"永续合约数量: {len(universe)}")
print("前 5 个:", [c["name"] for c in universe[:5]])

mids = info.all_mids()
print("ETH 中间价:", mids["ETH"], " BTC 中间价:", mids["BTC"])

永续合约数量: 212
前 5 个: ['SOL', 'APT', 'ATOM', 'BTC', 'ETH']


ETH 中间价: 2373.95  BTC 中间价: 80805.5


In [3]:
book = info.l2_snapshot("ETH")
bids, asks = book["levels"]
print("ETH 订单簿 (价格, 数量):")
print("  卖一~卖三:", [(l["px"], l["sz"]) for l in asks[:3]])
print("  买一~买三:", [(l["px"], l["sz"]) for l in bids[:3]])

ETH 订单簿 (价格, 数量):
  卖一~卖三: [('2374.1', '0.0046'), ('2374.2', '5.0'), ('2374.5', '0.0274')]
  买一~买三: [('2373.8', '5.0046'), ('2373.5', '0.2965'), ('2373.2', '0.296')]


In [4]:
import time

end = int(time.time() * 1000)
start = end - 24 * 60 * 60 * 1000
candles = info.candles_snapshot("ETH", "1h", start, end)
df = pd.DataFrame(candles)[["t", "o", "h", "l", "c", "v"]]
df["t"] = pd.to_datetime(df["t"], unit="ms")
df.columns = ["time", "open", "high", "low", "close", "volume"]
df.tail()

,time,open,high,low,close,volume
20,2026-10-24 03:00:00,2381.9,2384.3,2376.3,2379.4,165.6631
21,2026-10-24 04:00:00,2379.1,2383.1,2373.9,2378.4,166.1248
22,2026-10-24 05:00:00,2378.2,2378.2,2370.0,2373.5,181.848
23,2026-10-24 06:00:00,2373.5,2377.2,2370.2,2375.9,286.8661
24,2026-10-24 07:00:00,2375.5,2379.8,2372.8,2374.0,107.1313


## 2. 账户查询

查询永续保证金概况与持仓，以及现货余额，确认水龙头的 mock USDC 已到账。

In [5]:
state = info.user_state(ADDRESS)
summary = state["marginSummary"]
print("账户权益 (USDC):", summary["accountValue"])
print("已用保证金:", summary["totalMarginUsed"])

positions = [p["position"] for p in state["assetPositions"]]
if positions:
    for p in positions:
        print(f"持仓 {p['coin']}: 数量 {p['szi']} 入场价 {p['entryPx']}")
else:
    print("当前无永续持仓")

spot = info.spot_user_state(ADDRESS)
print("现货余额:", [(b["coin"], b["total"]) for b in spot["balances"]])

账户权益 (USDC): 0.0
已用保证金: 0.0
当前无永续持仓


现货余额: [('USDC', '998.988443')]


## 3. 限价挂单与撤单

以中间价的 50% 下一笔 ETH GTC 限价买单，这个价格远离市场不会成交，用于演示挂单、查询、撤单的完整流程。

数量按 `szDecimals` 取整，价格需满足永续合约的 tick 规则：最多 5 位有效数字，且小数位不超过 `6 - szDecimals`。

In [6]:
eth_meta = next(c for c in universe if c["name"] == "ETH")
sz_decimals = eth_meta["szDecimals"]


def round_px(px: float) -> float:
    """永续价格: 5 位有效数字且最多 6 - szDecimals 位小数。"""
    px = float(f"{px:.5g}")
    return round(px, 6 - sz_decimals)


mid = float(mids["ETH"])
limit_px = round_px(mid * 0.7)          # 远低于市价，挂单不会成交
sz = round(12 / limit_px, sz_decimals)  # 名义价值高于 10 USDC 最小值

print(f"挂单: 买入 {sz} ETH @ {limit_px}  (名义 {sz * limit_px:.2f} USDC)")
res = exchange.order("ETH", True, sz, limit_px, {"limit": {"tif": "Gtc"}})
assert res["status"] == "ok", res
status = res["response"]["data"]["statuses"][0]
assert "resting" in status, status  # 确认已挂单而非被拒
oid = status["resting"]["oid"]
print("挂单成功, oid =", oid)

挂单: 买入 0.0072 ETH @ 1661.8  (名义 11.96 USDC)


挂单成功, oid = 63547007936


In [7]:
open_before = info.open_orders(ADDRESS)
resting = [o for o in open_before if o["oid"] == oid]
print("当前挂单数:", len(open_before))
print("本次订单:", resting[0] if resting else "(已不在挂单列表)")

cancel_res = exchange.cancel("ETH", oid)
assert cancel_res["status"] == "ok", cancel_res
open_after = info.open_orders(ADDRESS)
print("撤单后挂单数:", len(open_after))
assert not any(o["oid"] == oid for o in open_after), "订单未被撤销"
print("撤单成功")

当前挂单数: 1
本次订单: {'coin': 'ETH', 'side': 'B', 'limitPx': '1661.8', 'sz': '0.0072', 'oid': 63547007936, 'timestamp': 1792828162202, 'origSz': '0.0072'}


撤单后挂单数: 0
撤单成功


## 4. 市价开仓与平仓

用 `market_open` 市价买入最小数量的 ETH，查询成交记录，再用 `market_close` 平仓，最后确认持仓归零。会真实成交，产生手续费与滑点。

In [8]:
open_sz = round(12 / mid, sz_decimals)
print(f"市价开多 {open_sz} ETH")
open_res = exchange.market_open("ETH", True, open_sz)
assert open_res["status"] == "ok", open_res
for s in open_res["response"]["data"]["statuses"]:
    if "filled" in s:
        f = s["filled"]
        print(f"成交: {f['totalSz']} ETH @ {f['avgPx']}")

市价开多 0.0051 ETH


成交: 0.0051 ETH @ 2374.1


In [9]:
fills = info.user_fills(ADDRESS)
print("最近成交:")
for f in fills[:3]:
    print(f"  {f['coin']} {f['dir']} {f['sz']} @ {f['px']}  手续费 {f['fee']}")

state_mid = info.user_state(ADDRESS)
pos = [p["position"] for p in state_mid["assetPositions"]]
print("平仓前持仓:", [(p["coin"], p["szi"]) for p in pos] or "无")

最近成交:
  ETH Open Long 0.0005 @ 2374.2  手续费 0.000534
  ETH Open Long 0.0046 @ 2374.1  手续费 0.004914
  ETH Close Long 0.0005 @ 2374.0  手续费 0.000534


平仓前持仓: [('ETH', '0.0051')]


In [10]:
close_res = exchange.market_close("ETH")
assert close_res["status"] == "ok", close_res
print("平仓请求已提交")

state_after = info.user_state(ADDRESS)
pos_after = [p["position"] for p in state_after["assetPositions"] if float(p["position"]["szi"]) != 0]
print("平仓后 ETH 持仓:", [(p["coin"], p["szi"]) for p in pos_after] or "已清空")
assert not any(p["coin"] == "ETH" for p in pos_after), "ETH 持仓未清空"
print("演示完成")

平仓请求已提交


平仓后 ETH 持仓: 已清空
演示完成
